# Group Stage

In [35]:
import numpy as np
import pandas as pd
import math

def predict_lambdas(beta, row):
    xH = np.array([1, row["home_rank"], row["home_gdp"], row["home_pop"], row["away_rank"], row["away_gdp"], row["away_pop"], 1])
    xA = np.array([1, row["away_rank"], row["away_gdp"], row["away_pop"], row["home_rank"], row["home_gdp"], row["home_pop"], 0])

    return float(np.exp(beta @ xH)), float(np.exp(beta @ xA))

def simulate_match(beta, row):
    lambdaH, lambdaA = predict_lambdas(beta, row)
    if lambdaH > lambdaA:
        lambdaH *= 2
    else:
        lambdaA *= 2

    home_goals = min(np.random.poisson(lambdaH), 6)
    away_goals = min(np.random.poisson(lambdaA), 6)


    return home_goals, away_goals

def simulate_group_stage(df_matches, beta):
    results = []

    for _, row in df_matches.iterrows():
        hs, as_ = simulate_match(beta, row)
        results.append({**row.to_dict(), "home_score": hs, "away_score": as_})

    return pd.DataFrame(results)


def build_group_tables(df_results):
    group_tables = {}

    for group in df_results["group"].unique():
        group_df = df_results[df_results["group"] == group]

        standings = {}

        for _, row in group_df.iterrows():
            ht, at = row["home_team"], row["away_team"]
            hs, as_ = row["home_score"], row["away_score"]

            for team in [ht, at]:
                if team not in standings:
                    standings[team] = {"points":0, "gd":0, "gf":0, "ga":0}

            standings[ht]["gf"] += hs
            standings[ht]["ga"] += as_
            standings[ht]["gd"] += hs - as_
            standings[at]["gf"] += as_
            standings[at]["ga"] += hs
            standings[at]["gd"] += as_ - hs

            if hs > as_:
                standings[ht]["points"] += 3
            elif hs < as_:
                standings[at]["points"] += 3
            else:
                standings[ht]["points"] += 1
                standings[at]["points"] += 1

        table = pd.DataFrame.from_dict(standings, orient="index")
        table["team"] = table.index
        table["group"] = group
        table = table.sort_values(by=["points", "gd", "gf"], ascending=False).reset_index(drop=True)
        group_tables[group] = table

    return group_tables

def extract_qualifiers(group_tables):
    top_two = []
    third_place = []

    for group, table in group_tables.items():
        group_letter = group.split()[-1]

        for i in range(2):
            row = table.iloc[i]
            top_two.append({"team": row["team"], "group": group_letter, "rank": f"{group_letter}{i+1}", "points": row["points"], "gd": row["gd"], "gf": row["gf"], "ga": row["ga"]})

        row = table.iloc[2]
        third_place.append({"team": row["team"], "group": group_letter, "rank": f"{group_letter}3", "points": row["points"], "gd": row["gd"], "gf": row["gf"], "ga": row["ga"]})

    df_top_two = pd.DataFrame(top_two)
    df_third = pd.DataFrame(third_place).sort_values(by=["points", "gd", "gf"], ascending=False).reset_index(drop=True)
    best_third = df_third.head(8)

    return df_top_two, df_third, best_third


def run_full_simulation(df_matches, beta):
    df_results = simulate_group_stage(df_matches, beta)
    group_tables = build_group_tables(df_results)
    top_two_df, third_df, best_third_df = extract_qualifiers(group_tables)
    return df_results, group_tables, top_two_df, third_df, best_third_df


beta = [ 0.00652995,  0.04557891, -0.00094763,  0.0011522,   0.01318271, -0.00819825,
 -0.00416648,  0.00915104]

df_matches = pd.read_csv("../Data/current_tournament_with_indicators.csv")

def normalize(col):
    std = col.std()
    return (col - col.mean()) / std if std != 0 else col * 0
 
df_matches["home_gdp"]  = normalize(np.log(df_matches["home_gdp"] + 1))
df_matches["away_gdp"]  = normalize(np.log(df_matches["away_gdp"] + 1))
df_matches["home_pop"]  = normalize(np.log(df_matches["home_pop"] + 1))
df_matches["away_pop"]  = normalize(np.log(df_matches["away_pop"] + 1))
df_matches["home_rank"] = normalize(-np.log(df_matches["home_rank"]))
df_matches["away_rank"] = normalize(-np.log(df_matches["away_rank"]))

df_results, group_tables, top_two_df, third_df, best_third_df = run_full_simulation(df_matches, beta)

# Knockout Stage

In [36]:
def get_team(rank, top_two_df):
    return top_two_df.loc[top_two_df["rank"] == rank, "team"].values[0]

def get_best_third(n):
    return best_third_df.iloc[n]["team"]

def simulate_knockout_match(beta, team1, team2, df_ref):
    row1 = df_ref[df_ref["home_team"] == team1].iloc[0]
    row2 = df_ref[df_ref["home_team"] == team2].iloc[0]

    fake_row = {"home_team": team1, "away_team": team2, "home_rank": row1["home_rank"], "home_gdp": row1["home_gdp"], "home_pop": row1["home_pop"], "away_rank": row2["home_rank"], "away_gdp": row2["home_gdp"], "away_pop": row2["home_pop"]}

    hs, as_ = simulate_match(beta, fake_row)

    while hs == as_:
        hs += np.random.binomial(1, 0.5)

    winner = team1 if hs > as_ else team2
    loser  = team2 if hs > as_ else team1

    return {"home_team": team1, "away_team": team2, "home_score": hs, "away_score": as_, "winner": winner, "loser": loser}


def run_full_knockout(df_results, top_two_df, best_third_df, beta, df_ref):
    matches = []
    
    R32 = {}
    R32[73] = simulate_knockout_match(beta, get_team("A2", top_two_df), get_team("B2", top_two_df), df_ref)
    R32[74] = simulate_knockout_match(beta, get_team("E1", top_two_df), get_best_third(0), df_ref)
    R32[75] = simulate_knockout_match(beta, get_team("F1", top_two_df), get_team("C2", top_two_df), df_ref)
    R32[76] = simulate_knockout_match(beta, get_team("C1", top_two_df), get_team("F2", top_two_df), df_ref)
    R32[77] = simulate_knockout_match(beta, get_team("I1", top_two_df), get_best_third(1), df_ref)
    R32[78] = simulate_knockout_match(beta, get_team("E2", top_two_df), get_team("I2", top_two_df), df_ref)
    R32[79] = simulate_knockout_match(beta, get_team("A1", top_two_df), get_best_third(2), df_ref)
    R32[80] = simulate_knockout_match(beta, get_team("L1", top_two_df), get_best_third(3), df_ref)
    R32[81] = simulate_knockout_match(beta, get_team("D1", top_two_df), get_best_third(4), df_ref)
    R32[82] = simulate_knockout_match(beta, get_team("G1", top_two_df), get_best_third(5), df_ref)
    R32[83] = simulate_knockout_match(beta, get_team("K2", top_two_df), get_team("L2", top_two_df), df_ref)
    R32[84] = simulate_knockout_match(beta, get_team("H1", top_two_df), get_team("J2", top_two_df), df_ref)
    R32[85] = simulate_knockout_match(beta, get_team("B1", top_two_df), get_best_third(6), df_ref)
    R32[86] = simulate_knockout_match(beta, get_team("J1", top_two_df), get_team("H2", top_two_df), df_ref)
    R32[87] = simulate_knockout_match(beta, get_team("K1", top_two_df), get_best_third(7), df_ref)
    R32[88] = simulate_knockout_match(beta, get_team("D2", top_two_df), get_team("G2", top_two_df), df_ref)

    R16 = {}
    R16[89] = simulate_knockout_match(beta, R32[74]["winner"], R32[77]["winner"], df_ref)
    R16[90] = simulate_knockout_match(beta, R32[73]["winner"], R32[75]["winner"], df_ref)
    R16[91] = simulate_knockout_match(beta, R32[76]["winner"], R32[78]["winner"], df_ref)
    R16[92] = simulate_knockout_match(beta, R32[79]["winner"], R32[80]["winner"], df_ref)
    R16[93] = simulate_knockout_match(beta, R32[83]["winner"], R32[84]["winner"], df_ref)
    R16[94] = simulate_knockout_match(beta, R32[81]["winner"], R32[82]["winner"], df_ref)
    R16[95] = simulate_knockout_match(beta, R32[86]["winner"], R32[88]["winner"], df_ref)
    R16[96] = simulate_knockout_match(beta, R32[85]["winner"], R32[87]["winner"], df_ref)

    QF = {}
    QF[97]  = simulate_knockout_match(beta, R16[89]["winner"], R16[90]["winner"], df_ref)
    QF[98]  = simulate_knockout_match(beta, R16[93]["winner"], R16[94]["winner"], df_ref)
    QF[99]  = simulate_knockout_match(beta, R16[91]["winner"], R16[92]["winner"], df_ref)
    QF[100] = simulate_knockout_match(beta, R16[95]["winner"], R16[96]["winner"], df_ref)

    SF = {}
    SF[101] = simulate_knockout_match(beta, QF[97]["winner"], QF[98]["winner"], df_ref)
    SF[102] = simulate_knockout_match(beta, QF[99]["winner"], QF[100]["winner"], df_ref)

    third = simulate_knockout_match(beta, SF[101]["loser"], SF[102]["loser"], df_ref)
    final = simulate_knockout_match(beta, SF[101]["winner"], SF[102]["winner"], df_ref)


    def flatten(stage_dict, stage_name):
        return [{"stage": stage_name, "match_id": k, **v} for k, v in stage_dict.items()]

    all_knockout = (flatten(R32, "R32") + flatten(R16, "R16") + flatten(QF, "Quarterfinal") + flatten(SF, "Semifinal") + [{"stage": "Third Place", "match_id": 103, **third}] + [{"stage": "Final", "match_id": 104, **final}])
    df_knockout = pd.DataFrame(all_knockout)

    df_group = df_results.copy()
    df_group["stage"] = "Group"
    df_group["match_id"] = range(len(df_group))

    full_tournament = pd.concat([df_group, df_knockout], ignore_index=True)
    full_tournament[["stage", "group", "home_team", "away_team", "home_score", "away_score"]].to_csv("full_tournament_results.csv", index=False)

    return full_tournament

full_results = run_full_knockout(df_results, top_two_df, best_third_df, beta, df_matches)

In [34]:


def monte_carlo(n_sim, df_matches, beta):
    winners = {}

    for sim in range(n_sim):
        df_results, group_tables, top_two_df, third_df, best_third_df = run_full_simulation(df_matches, beta)
        full_results = run_full_knockout(df_results,top_two_df,best_third_df,beta,df_matches)
        champion = full_results[full_results["stage"] == "Final"]["winner"].values[0]
        if champion in winners:
            winners[champion] += 1
        else:
            winners[champion] = 1
    results = pd.DataFrame({"team": winners.keys(),"titles": winners.values()})
    results["probability"] = results["titles"] / n_sim
    results = results.sort_values("probability", ascending=False)
    return results

results = monte_carlo(1000, df_matches, beta)

print(results)

            team  titles  probability
0         france     235        0.235
3          spain     156        0.156
4         brazil     116        0.116
1      argentina      97        0.097
5        germany      83        0.083
9        england      66        0.066
7       portugal      34        0.034
12   netherlands      30        0.030
8            usa      26        0.026
2         mexico      22        0.022
6        morocco      19        0.019
11   south korea      15        0.015
10       belgium      10        0.010
13   switzerland      10        0.010
15         japan      10        0.010
21      colombia      10        0.010
18       senegal       7        0.007
23       türkiye       6        0.006
19        canada       6        0.006
17       czechia       5        0.005
22       croatia       5        0.005
14       uruguay       4        0.004
26        sweden       4        0.004
20      dr congo       3        0.003
16       ecuador       3        0.003
24   ivory c